# MiniCode — Fine-Tuning Supervisionado (SFT + QLoRA) e Avaliação Nível 1

Pipeline reprodutível de treinamento e avaliação do modelo estudante **Qwen/Qwen2.5-Coder-1.5B-Instruct** sobre as trajetórias filtradas do professor (**Qwen/Qwen2.5-14B-Instruct-AWQ**), estruturado na sequência:
1. **Dependências e Ambiente** (`git clone` + `pip install -e .`)
2. **Dataset e Configuração** (`datasets/sft_jsonl/{train,val,test}.jsonl`)
3. **Tokenização e Máscara *Completion-Only***
4. **Modelo Base (4-bit NF4 / FP16 nativo para GPU T4)**
5. **Configuração LoRA / QLoRA** (derivação exata dos `18.464.768` parâmetros)
6. **Treino (`SFTTrainer`)**
7. **Gráficos (`loss` de treino e validação × época)**
8. **Inferência no `test.jsonl` (adapter desligado vs. ligado)**
9. **Avaliação Nível 1 (reutilizando o código do repositório)**


## 1. Dependências e Ambiente (`git clone` + `pip install -e .`)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

REPO_URL = "https://github.com/jp-huppes/MiniCode.git"
REPO_DIR = Path("/content/MiniCode")

for cand in [Path.cwd(), Path.cwd().parent, REPO_DIR]:
    if (cand / "pyproject.toml").exists() and (cand / "datasets" / "sft_jsonl" / "train.jsonl").exists():
        os.chdir(cand)
        break
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("Working dir:", Path.cwd())


In [ ]:
%pip install -q -e .
%pip install -q "transformers>=4.46.0" "peft>=0.13.0" "trl>=0.12.0" "bitsandbytes>=0.44.0" "accelerate>=1.0.0" "datasets>=3.0.0" matplotlib


In [ ]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM total: {vram_gb:.2f} GB")

for pkg in ["transformers", "peft", "trl", "bitsandbytes", "datasets"]:
    mod = __import__(pkg)
    print(f"{pkg}: {getattr(mod, '__version__', 'ok')}")


## 2. Dataset e Configuração


In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from src.dataset.build_sft_dataset import DEFAULT_SYSTEM_PROMPT, PROMPT_VERSION

@dataclass
class TrainConfig:
    model_id: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    train_file: str = "datasets/sft_jsonl/train.jsonl"
    val_file: str = "datasets/sft_jsonl/val.jsonl"
    test_file: str = "datasets/sft_jsonl/test.jsonl"
    output_dir: str = "checkpoints/qwen25_coder_1_5b_lora"
    results_dir: str = "results"
    prompt_version: str = PROMPT_VERSION
    system_prompt: str = DEFAULT_SYSTEM_PROMPT
    seed: int = 42
    max_seq_length: int = 2048

    # LoRA / QLoRA
    use_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # SFT (batch efetivo = 1 * 16 = 16)
    num_train_epochs: int = 3
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16

cfg = TrainConfig()

def load_jsonl_records(path_str: str) -> list[dict]:
    p = Path(path_str)
    if not p.exists():
        raise FileNotFoundError(f"Split não encontrado: {p.resolve()}")
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows = load_jsonl_records(cfg.train_file)
val_rows = load_jsonl_records(cfg.val_file)
test_rows = load_jsonl_records(cfg.test_file)

assert len(train_rows) > 0 and len(val_rows) > 0 and len(test_rows) > 0
for r in train_rows[:3]:
    assert r["messages"][0]["content"] == cfg.system_prompt, "Divergência de system_prompt!"

print(f"Prompt version: {cfg.prompt_version}")
print(f"Exemplos carregados -> train: {len(train_rows)}, val: {len(val_rows)}, test: {len(test_rows)}")


## 3. Tokenização e Máscara de *Completion-Only*

Aplicamos o *chat template* do Qwen2.5 e mascaramos com `-100` todos os tokens anteriores à resposta do `assistant`, garantindo que a *loss* supervisionada incida exclusivamente sobre a ação JSON alvo.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_completion_only_example(row: dict) -> dict:
    messages = row.get("messages", [])
    if len(messages) < 3:
        raise ValueError("Esperado ao menos 3 mensagens: system, user e assistant.")

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(
        prompt_text,
        truncation=True,
        max_length=cfg.max_seq_length,
        add_special_tokens=False,
    )["input_ids"]
    full_ids = tokenizer(
        full_text,
        truncation=True,
        max_length=cfg.max_seq_length,
        add_special_tokens=False,
    )["input_ids"]

    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    assert any(lbl != -100 for lbl in labels), "Exemplo totalmente mascarado após truncamento!"

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

train_ds = Dataset.from_list(train_rows).map(
    build_completion_only_example,
    remove_columns=list(train_rows[0].keys()),
)
val_ds = Dataset.from_list(val_rows).map(
    build_completion_only_example,
    remove_columns=list(val_rows[0].keys()),
)

print(f"Datasets tokenizados -> train_ds: {len(train_ds)}, val_ds: {len(val_ds)}")
print(f"Tokens (ex. 0) -> total={len(train_ds[0]['input_ids'])}, supervisionados={sum(x != -100 for x in train_ds[0]['labels'])}")


## 4. Modelo Base (QLoRA 4-bit NF4 / FP16 Nativo para T4)

Carregamos `Qwen/Qwen2.5-Coder-1.5B-Instruct` quantizado em **4-bit NF4** com `bnb_4bit_compute_dtype=torch.float16` e convertemos qualquer tensor residual em `bfloat16` para `float16`/`float32`, assegurando compatibilidade nativa com GPUs Turing (Tesla T4, *compute capability* 7.5).


In [ ]:
import gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

for _v in ["model", "trainer"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

print("Modelo base carregado em 4-bit NF4:", cfg.model_id)


## 5. Configuração LoRA / QLoRA

### Fundamentação Matemática e Contagem Exata no Qwen2.5-1.5B

Na adaptação de baixo posto (LoRA), cada matriz congelada $W \in \mathbb{R}^{d_{	ext{out}} 	imes d_{	ext{in}}}$ recebe uma atualização fatorada de posto $r \ll \min(d_{	ext{in}}, d_{	ext{out}})$:
$$W' = W + 
rac{lpha}{r} B A, \qquad A \in \mathbb{R}^{r 	imes d_{	ext{in}}}, \quad B \in \mathbb{R}^{d_{	ext{out}} 	imes r}$$
totalizando $r(d_{	ext{in}} + d_{	ext{out}})$ parâmetros treináveis por matriz.

Conforme o `config.json` do **Qwen2.5-1.5B / Qwen2.5-Coder-1.5B-Instruct**:
* $L = 28$ camadas (`num_hidden_layers = 28`)
* $d = 1536$ (`hidden_size = 1536`)
* $n_h = 12$ cabeças Q e $n_{kv} = 2$ cabeças KV (GQA), logo $d_{	ext{head}} = 128$ e $d_{kv} = 2 	imes 128 = 256$
* $d_{	ext{ff}} = 8960$ (`intermediate_size = 8960`)
* $r = 16$ nas 7 projeções lineares por bloco:

| Módulo | Dimensões ($d_{	ext{in}} 	o d_{	ext{out}}$) | Parâmetros por Camada ($r = 16$) |
| :--- | :---: | :---: |
| `q_proj` | $1536 	o 1536$ | $16 	imes (1536 + 1536) = 49.152$ |
| `k_proj` | $1536 	o 256$ | $16 	imes (1536 + 256) = 28.672$ |
| `v_proj` | $1536 	o 256$ | $16 	imes (1536 + 256) = 28.672$ |
| `o_proj` | $1536 	o 1536$ | $16 	imes (1536 + 1536) = 49.152$ |
| `gate_proj` | $1536 	o 8960$ | $16 	imes (1536 + 8960) = 167.936$ |
| `up_proj` | $1536 	o 8960$ | $16 	imes (1536 + 8960) = 167.936$ |
| `down_proj` | $8960 	o 1536$ | $16 	imes (8960 + 1536) = 167.936$ |
| **Total por camada** | — | **$659.456$** |
| **Total no modelo ($28$ camadas)** | — | **$28 	imes 659.456 = 18.464.768$** |


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.target_modules),
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)

for mod in model.modules():
    if getattr(mod, "compute_dtype", None) == torch.bfloat16:
        mod.compute_dtype = torch.float16
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)
    elif p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for b in model.buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)
model.config.dtype = torch.float16
model.config.torch_dtype = torch.float16

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable_params == 18_464_768, f"Esperado 18464768, obtido {trainable_params}"

print("Dtypes ativos no modelo:", {p.dtype for p in model.parameters()})
model.print_trainable_parameters()


## 6. Treino (`SFTTrainer`)


In [ ]:
from transformers import DataCollatorForSeq2Seq
from trl import SFTConfig, SFTTrainer

sft_args = SFTConfig(
    output_dir=cfg.output_dir,
    max_length=cfg.max_seq_length,
    optim="paged_adamw_8bit",
    num_train_epochs=cfg.num_train_epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    fp16=False,
    bf16=False,
    seed=cfg.seed,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_num_proc=1,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
)

train_result = trainer.train()

final_dir = Path(cfg.output_dir) / "final_adapter"
final_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

metrics_out = Path(cfg.output_dir) / "training_metrics.json"
metrics_out.write_text(
    json.dumps(
        {
            "train_metrics": train_result.metrics,
            "log_history": trainer.state.log_history,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Treino concluído! Adapter salvo em:", final_dir.resolve())
print("Métricas de treino:", train_result.metrics)
print("Histórico salvo em:", metrics_out.resolve())


## 7. Gráficos de Loss de Treino e Validação por Época

Como `logging_steps=5` registra a *loss* de treino a cada 5 passos de otimização, agrupamos cada registro na sua época inteira correspondente ($\lceil 	ext{epoch} 
ceil$) e plotamos a média por época ao lado da *loss* de validação (`eval_loss`).


In [ ]:
import math
from collections import defaultdict
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_loss_by_epoch: dict[int, list[float]] = defaultdict(list)
val_loss_by_epoch: dict[int, float] = {}

for entry in log_history:
    if "loss" in entry and "epoch" in entry:
        ep = max(1, int(math.ceil(float(entry["epoch"]) - 1e-9)))
        train_loss_by_epoch[ep].append(float(entry["loss"]))
    if "eval_loss" in entry and "epoch" in entry:
        ep = max(1, int(round(float(entry["epoch"]))))
        val_loss_by_epoch[ep] = float(entry["eval_loss"])

epochs_train = sorted(train_loss_by_epoch.keys())
train_means = [sum(train_loss_by_epoch[e]) / len(train_loss_by_epoch[e]) for e in epochs_train]

epochs_val = sorted(val_loss_by_epoch.keys())
val_losses = [val_loss_by_epoch[e] for e in epochs_val]

print("Época | Train Loss (média passos) | Val Loss")
for ep in sorted(set(epochs_train) | set(epochs_val)):
    tr_str = f"{sum(train_loss_by_epoch[ep]) / len(train_loss_by_epoch[ep]):.4f}" if ep in train_loss_by_epoch else "-"
    va_str = f"{val_loss_by_epoch[ep]:.4f}" if ep in val_loss_by_epoch else "-"
    print(f"  {ep}   |          {tr_str}           |  {va_str}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs_train, train_means, marker="o", linewidth=2, color="#1f77b4", label="Train Loss (média/época)")
for ep in epochs_train:
    vals = train_loss_by_epoch[ep]
    axes[0].scatter([ep] * len(vals), vals, alpha=0.35, color="#1f77b4", s=28)
axes[0].set_title("Loss de Treino × Época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_xticks(epochs_train)
axes[0].grid(True, linestyle="--", alpha=0.4)
axes[0].legend()

axes[1].plot(epochs_val, val_losses, marker="s", linewidth=2, color="#d62728", label="Validation Loss")
best_ep = min(val_loss_by_epoch, key=val_loss_by_epoch.get)
axes[1].scatter([best_ep], [val_loss_by_epoch[best_ep]], color="#2ca02c", s=90, zorder=5, label=f"Melhor (Época {best_ep}: {val_loss_by_epoch[best_ep]:.4f})")
axes[1].set_title("Loss de Validação × Época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Validation Loss")
axes[1].set_xticks(epochs_val)
axes[1].grid(True, linestyle="--", alpha=0.4)
axes[1].legend()

plt.tight_layout()
plot_path = Path(cfg.output_dir) / "loss_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo em:", plot_path.resolve())


## 8. Inferência no `test.jsonl` (Adapter Desligado vs. Ligado)

Executamos geração determinística (`do_sample=False`) nos 36 exemplos do split `test.jsonl` em dois modos sobre o mesmo modelo carregado:
* **Adapter desligado (`with model.disable_adapter():`):** comportamento *zero-shot* do `Qwen2.5-Coder-1.5B-Instruct` base.
* **Adapter ligado:** comportamento do melhor *checkpoint* LoRA restaurado por `load_best_model_at_end=True`.


In [ ]:
trained_model = trainer.model
trained_model.eval()
trained_model.config.use_cache = True
tokenizer.padding_side = "left"

def run_test_inference(peft_model, rows: list[dict], max_new_tokens: int = 256) -> list[str]:
    predictions: list[str] = []
    device = next(peft_model.parameters()).device
    with torch.inference_mode():
        for idx, row in enumerate(rows, start=1):
            prompt_msgs = row["messages"][:-1]
            prompt_text = tokenizer.apply_chat_template(
                prompt_msgs,
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
            output_ids = peft_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            gen_ids = output_ids[0][inputs["input_ids"].shape[1]:]
            pred_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            predictions.append(pred_text)
    return predictions

with trained_model.disable_adapter():
    raw_preds_base = run_test_inference(trained_model, test_rows)

raw_preds_lora = run_test_inference(trained_model, test_rows)

results_dir = Path(cfg.results_dir)
results_dir.mkdir(parents=True, exist_ok=True)

def save_predictions_jsonl(path: Path, rows: list[dict], preds: list[str], system_tag: str) -> None:
    lines = []
    for row, pred in zip(rows, preds):
        record = {
            "system": system_tag,
            "task_id": row.get("task_id"),
            "run_id": row.get("run_id"),
            "fsm_state": row.get("fsm_state"),
            "reference": row.get("y") or json.loads(row["messages"][-1]["content"]),
            "raw_prediction": pred,
        }
        lines.append(json.dumps(record, ensure_ascii=False))
    path.write_text("
".join(lines) + "
", encoding="utf-8")

base_pred_path = results_dir / "test_predictions_base.jsonl"
lora_pred_path = results_dir / "test_predictions_lora.jsonl"
save_predictions_jsonl(base_pred_path, test_rows, raw_preds_base, "1.5B_base")
save_predictions_jsonl(lora_pred_path, test_rows, raw_preds_lora, "1.5B_lora")

print(f"Predições salvas -> {base_pred_path} ({len(raw_preds_base)}) | {lora_pred_path} ({len(raw_preds_lora)})")
print("Exemplo [0] Base:", raw_preds_base[0][:160])
print("Exemplo [0] LoRA:", raw_preds_lora[0][:160])


## 9. Avaliação Nível 1 (Reutilizando o Código do Repositório)

Reutilizamos diretamente `is_schema_valid`, `normalize_args` e `wilson_or_normal_ci95` de `src/eval/eval_level1.py`, junto com `evaluate_predictions` e `paired_bootstrap_delta_ci` de `scripts/eval_baselines.py` (bootstrap pareado por `run_id`, $10.000$ reamostragens), garantindo paridade total com a metodologia oficial do projeto.


In [ ]:
from src.eval.eval_level1 import is_schema_valid, normalize_args, wilson_or_normal_ci95
from scripts.eval_baselines import evaluate_predictions, paired_bootstrap_delta_ci

def compute_strict_level1_metrics(examples: list[dict], raw_responses: list[str]) -> dict:
    base_metrics = evaluate_predictions(examples, raw_responses)
    n = len(examples)
    schema_valid_count = 0
    strict_exact_count = 0

    for ex, raw in zip(examples, raw_responses):
        ref = ex.get("y") or json.loads(ex["messages"][-1]["content"])
        ref_tool = str(ref.get("tool", "")).strip()
        ref_args = normalize_args(ref.get("args") or {})

        ok_schema, parsed_obj = is_schema_valid(raw)
        if ok_schema and parsed_obj is not None:
            schema_valid_count += 1
            if parsed_obj["tool"] == ref_tool and normalize_args(parsed_obj["args"]) == ref_args:
                strict_exact_count += 1

    schema_rate = round(schema_valid_count / n, 4) if n else 0.0
    strict_exact_rate = round(strict_exact_count / n, 4) if n else 0.0

    return {
        **base_metrics,
        "schema_validity": schema_rate,
        "schema_validity_ci95": list(wilson_or_normal_ci95(schema_rate, n)),
        "strict_exact_action_acc": strict_exact_rate,
        "strict_exact_action_ci95": list(wilson_or_normal_ci95(strict_exact_rate, n)),
    }

eval_base = compute_strict_level1_metrics(test_rows, raw_preds_base)
eval_lora = compute_strict_level1_metrics(test_rows, raw_preds_lora)

bootstrap_lora_vs_base = paired_bootstrap_delta_ci(
    test_rows,
    eval_lora["per_example_correct"],
    eval_base["per_example_correct"],
    n_resamples=10000,
    seed=cfg.seed,
)

level1_report = {
    "prompt_version": cfg.prompt_version,
    "model_id": cfg.model_id,
    "adapter_dir": str(Path(cfg.output_dir) / "final_adapter"),
    "test_examples_count": len(test_rows),
    "systems": {
        "1.5B_base": {k: v for k, v in eval_base.items() if k != "per_example_correct"},
        "1.5B_lora": {k: v for k, v in eval_lora.items() if k != "per_example_correct"},
    },
    "comparisons": {
        "1.5B_lora_vs_1.5B_base": bootstrap_lora_vs_base,
    },
}

report_path = Path(cfg.results_dir) / "level1_colab_eval.json"
report_path.write_text(json.dumps(level1_report, ensure_ascii=False, indent=2), encoding="utf-8")

print("=== RESULTADOS NÍVEL 1 (TEST.JSONL, n=36) ===")
for sys_name in ["1.5B_base", "1.5B_lora"]:
    m = level1_report["systems"][sys_name]
    print(
        f"{sys_name:10s} | Tool Acc={m['acc']:.4f} | Schema Valid={m['schema_validity']:.4f} "
        f"| Exact Action={m['strict_exact_action_acc']:.4f} | F1-Macro={m['f1_macro']:.4f}"
    )
print("
Bootstrap pareado (1.5B_lora vs 1.5B_base):", bootstrap_lora_vs_base)
print("Relatório salvo em:", report_path.resolve())
